# 06 - Regression

Train and evaluate models for predicting numeric support-ticket outcomes.

In [3]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load Preprocessed Data
df = pd.read_csv('../data/customer_tickets_cleaned.csv')
df.columns

Index(['ticket_id', 'customer_name', 'customer_email', 'product_purchased',
       'date_of_purchase', 'ticket_type', 'ticket_subject',
       'ticket_description', 'ticket_priority', 'first_response_time',
       'time_to_resolution', 'customer_satisfaction_rating',
       'clean_description', 'priority_encoded', 'satisfaction_category',
       'word_count', 'purchase_age_days', 'sentiment_score'],
      dtype='object')

In [4]:
# Ensure target column is in days (if your dataset has hours, convert by dividing by 24)
# Example: df['time_to_resolution_days'] = df['Time to Resolution'] / 24.0
# Assuming 'time_to_resolution_days' is your target variable:
TARGET = 'time_to_resolution'  # Adjust column name if needed

# 2. Define Feature Subsets matching your pipeline
text_features = ['clean_description', 'ticket_subject']
categorical_features = ['ticket_type', 'satisfaction_category']
numeric_features = ['sentiment_score']

# Drop missing values in features/target
feature_cols = text_features + categorical_features + numeric_features
df = df.dropna(subset=feature_cols + [TARGET])

X = df[feature_cols]
y = df[TARGET]

# 3. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Construct Preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('desc_tfidf', TfidfVectorizer(max_features=2000, stop_words='english'), 'clean_description'),
        ('subj_tfidf', TfidfVectorizer(max_features=500, stop_words='english'), 'ticket_subject'),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('num_scaler', StandardScaler(), numeric_features)
    ]
)

# 5. Create Full Pipeline with Regressor (e.g., RandomForestRegressor or Ridge)
reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 6. Train the Model
reg_pipeline.fit(X_train, y_train)

# 7. Evaluate Performance
y_pred = reg_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Custom Accuracy Metric: % predictions within +/- 0.5 days (12 hours)
within_half_day = np.mean(np.abs(y_test - y_pred) <= 0.5) * 100

print(f"--- Model Evaluation ---")
print(f"MAE  (Mean Absolute Error): {mae:.2f} days")
print(f"RMSE (Root Mean Sq Error):  {rmse:.2f} days")
print(f"R² Score:                   {r2:.4f}")
print(f"Accuracy (within ±12 hrs): {within_half_day:.2f}%")

# 8. Save Model Artifact
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
# joblib.dump(reg_pipeline, MODEL_DIR / 'RandomForest_resolution_time.pkl')
print(f"\n✅ Model saved to {MODEL_DIR / 'RandomForest_resolution_time.pkl'}")

--- Model Evaluation ---
MAE  (Mean Absolute Error): 33.95 days
RMSE (Root Mean Sq Error):  43.55 days
R² Score:                   -0.3252
Accuracy (within ±12 hrs): 0.00%

✅ Model saved to ..\models\RandomForest_resolution_time.pkl


In [8]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load Preprocessed Data
df = pd.read_csv('../data/customer_tickets_cleaned.csv')

# Use 'Time to Resolution' directly in HOURS (ensure positive values)
TARGET = 'time_to_resolution'
df = df[df[TARGET] > 0].copy()

# 2. Define Features
text_features = ['clean_description', 'ticket_subject']
categorical_features = ['ticket_type', 'satisfaction_category']
numeric_features = ['sentiment_score']

feature_cols = text_features + categorical_features + numeric_features
df = df.dropna(subset=feature_cols + [TARGET])

X = df[feature_cols]

# Log-transform target (in hours) to handle right-skewness and outliers
y_log = np.log1p(df[TARGET])

# 3. Train / Test Split
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# 4. Preprocessor Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('desc_tfidf', TfidfVectorizer(max_features=1000, stop_words='english'), 'clean_description'),
        ('subj_tfidf', TfidfVectorizer(max_features=300, stop_words='english'), 'ticket_subject'),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('num_scaler', StandardScaler(), numeric_features)
    ],sparse_threshold=0  # <-- Forces output to be a dense numpy array
)

# Transform features
X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

# 5. Train Regressor
model = HistGradientBoostingRegressor(random_state=42)
model.fit(X_train_trans, y_train_log)

# 6. Predict and convert back from log scale back to HOURS (expm1)
y_pred_log = model.predict(X_test_trans)
y_pred_hours = np.expm1(y_pred_log)
y_test_hours = np.expm1(y_test_log)

# 7. Evaluate Performance in HOURS
mae_hours = mean_absolute_error(y_test_hours, y_pred_hours)
rmse_hours = np.sqrt(mean_squared_error(y_test_hours, y_pred_hours))
r2 = r2_score(y_test_hours, y_pred_hours)

# Practical Accuracy Metrics (Tolerance Windows)
acc_4hrs = np.mean(np.abs(y_test_hours - y_pred_hours) <= 4.0) * 100
acc_12hrs = np.mean(np.abs(y_test_hours - y_pred_hours) <= 12.0) * 100
acc_24hrs = np.mean(np.abs(y_test_hours - y_pred_hours) <= 24.0) * 100

print(f"--- Model Evaluation (Target: Hours) ---")
print(f"MAE  (Mean Absolute Error): {mae_hours:.2f} hours")
print(f"RMSE (Root Mean Sq Error):  {rmse_hours:.2f} hours")
print(f"R² Score:                   {r2:.4f}")
print(f"----------------------------------------")
print(f"Accuracy within ±4 hours:   {acc_4hrs:.2f}%")
print(f"Accuracy within ±12 hours:  {acc_12hrs:.2f}%")
print(f"Accuracy within ±24 hours:  {acc_24hrs:.2f}%")

# 8. Bundle into a single Pipeline and Save Artifact
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', model)
])

MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
# joblib.dump(final_pipeline, MODEL_DIR / 'resolution_time_regressor.pkl')
print(f"\n✅ Pipeline saved to {MODEL_DIR / 'resolution_time_regressor.pkl'}")

--- Model Evaluation (Target: Hours) ---
MAE  (Mean Absolute Error): 33.88 hours
RMSE (Root Mean Sq Error):  42.95 hours
R² Score:                   -0.2885
----------------------------------------
Accuracy within ±4 hours:   8.33%
Accuracy within ±12 hours:  16.67%
Accuracy within ±24 hours:  47.92%

✅ Pipeline saved to ..\models\resolution_time_regressor.pkl


In [11]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load Data
df = pd.read_csv('../data/customer_tickets_cleaned.csv')

TARGET = 'time_to_resolution'
df = df[df[TARGET] > 0].copy()

# 2. Select Core Signal Features
text_features = ['clean_description', 'ticket_subject']
categorical_features = ['ticket_type', 'satisfaction_category', 'ticket_priority']
# Include key signal features identified in correlation
numeric_features = ['purchase_age_days', 'first_response_time', 'sentiment_score']

feature_cols = text_features + categorical_features + numeric_features
df = df.dropna(subset=feature_cols + [TARGET])

X = df[feature_cols]
y_log = np.log1p(df[TARGET])  # Log transform to handle skewness

# 3. Train/Test Split
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# 4. Preprocessor Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('desc_tfidf', TfidfVectorizer(max_features=100, stop_words='english'), 'clean_description'),
        ('subj_tfidf', TfidfVectorizer(max_features=50, stop_words='english'), 'ticket_subject'),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('num_scaler', StandardScaler(), numeric_features)
    ]
)

# 5. Build Pipeline with Ridge Regression (linear model prevents overfitting on TF-IDF)
reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=10.0))
])

# 6. Fit & Evaluate
reg_pipeline.fit(X_train, y_train_log)

y_pred_log = reg_pipeline.predict(X_test)
y_pred_hours = np.expm1(y_pred_log)
y_test_hours = np.expm1(y_test_log)

# Metrics in Hours
mae_hours = mean_absolute_error(y_test_hours, y_pred_hours)
rmse_hours = np.sqrt(mean_squared_error(y_test_hours, y_pred_hours))
r2 = r2_score(y_test_hours, y_pred_hours)

print(f"--- Model Evaluation (Target: Hours) ---")
print(f"MAE  (Mean Absolute Error): {mae_hours:.2f} hours")
print(f"RMSE (Root Mean Sq Error):  {rmse_hours:.2f} hours")
print(f"R² Score:                   {r2:.4f}")

# 7. Save Artifact
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(reg_pipeline, MODEL_DIR / 'resolution_time_regressor.pkl')
print(f"\n✅ Regressor pipeline saved to {MODEL_DIR / 'resolution_time_regressor.pkl'}")

--- Model Evaluation (Target: Hours) ---
MAE  (Mean Absolute Error): 27.90 hours
RMSE (Root Mean Sq Error):  30.71 hours
R² Score:                   0.1778

✅ Regressor pipeline saved to ..\models\resolution_time_regressor.pkl
